# Task 1: Data Cleaning and Preprocessing

**Goal:**: To improve the quality, consistency, and reliability of Netflix catalogue data by identifying and handling missing values, duplicates, inconsistent text formats, and data type issues.

**Workflow**
1. Import the dataset with Pandas
2. Identify missing and duplicate records
3. Handle null values and inconsistent formats
4. Transform categorical and date-related columns
5. Save a clean dataset for further analysis

## Step 1: Import the dataset

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

raw = pd.read_csv("Dataset.csv")
df = raw.copy()   # keep the raw data untouched for before/after comparison

print("Shape:", df.shape)
df.head()

Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8790,8790,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8790,2,Movie,6126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8790,8787,9-Feb,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,8790,4528,Not Given,2588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,8790,86,United States,3240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8790,1713,1/1/2020,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8790.0,NaN,NaN,NaN,2014.183163,8.825466,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8790,14,TV-MA,3205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8790,220,1 Season,1791,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listed_in,8790,513,"Dramas, International Movies",362,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Observations**
- 8,790 rows and 10 columns. One row per title.
- Every column except `release_year` is stored as text, including `date_added` and `duration`, which need converting.
- `type` has only two values (Movie, TV Show), so it is a natural categorical column.

## Step 2: Identify missing and duplicate records

In [5]:
# Standard null check
df.isnull().sum()

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

The standard check reports **zero** nulls. That does not mean the data is complete. Missing values can be hidden as placeholder text, in the next cell we search for common placeholders.

In [6]:
text_cols = ["show_id", "type", "title", "director", "country",
             "date_added", "rating", "duration", "listed_in"]
placeholders = ["Not Given", "Unknown", "N/A", "NA", "None", "", "-"]

hidden_missing = {
    col: int(df[col].astype(str).str.strip().isin(placeholders).sum())
    for col in text_cols
}
hidden_missing = pd.Series(hidden_missing)
hidden = hidden_missing[hidden_missing > 0].to_frame("count")
hidden["percent"] = (hidden["count"] / len(df) * 100).round(1)
hidden

,count,percent
title,1,0.0
director,2588,29.4
country,287,3.3


**Finding:** missing values are hidden as the text `"Not Given"`:
- `director`: about 29% of rows
- `country`: about 3% of rows

These would silently distort later analysis (for example, "Not Given" would appear as the most common director).

The single match in `title` is the 2011 film *Unknown*, a real title, so it is left as is.

In [7]:
# Exact duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

# Duplicate IDs
print("Duplicate show_id values:", df["show_id"].duplicated().sum())

# Duplicates ignoring the ID column (same title entered twice under different IDs)
content_cols = [c for c in df.columns if c != "show_id"]
print("Duplicate records ignoring show_id:", df.duplicated(subset=content_cols).sum())

Exact duplicate rows: 0
Duplicate show_id values: 0
Duplicate records ignoring show_id: 3


In [8]:
dupes = df[df.duplicated(subset=content_cols, keep=False)].sort_values("title")
dupes

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
393,s3997,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Not Given,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"


**Finding:** no exact duplicates, but **3 titles appear twice** with identical details and different `show_id` values. These are true duplicates and should be removed. (A fourth duplicate is hidden by invisible whitespace and shows up after Step 3a.)

Also note the titles `15-Aug`, `22-Jul` and `9-Feb`. These look like titles that were auto-converted to dates by a spreadsheet program (for example "15 August" became "15-Aug"). This is handled in Step 3.

## Step 3: Handle null values and inconsistent formats

### 3a. Clean whitespace in text columns
Some text contains non-breaking spaces (a hidden character often copied from web pages) and trailing spaces. These make identical values look different, for example `"Consequences "` vs `"Consequences"`. This is done **before** removing duplicates so that hidden duplicates are caught.

In [9]:
str_cols = ["show_id", "type", "title", "director", "country",
            "date_added", "rating", "duration", "listed_in"]

changed = 0
for col in str_cols:
    cleaned = df[col].astype(str).str.replace("\u00a0", " ", regex=False).str.strip()
    changed += (cleaned != df[col]).sum()
    df[col] = cleaned

print("Cells with extra whitespace fixed:", changed)

Cells with extra whitespace fixed: 5


### 3b. Remove duplicate records
Recheck duplicates now that whitespace is clean, then keep the first occurrence of each.

In [10]:
print("Duplicates after whitespace fix:", df.duplicated(subset=content_cols).sum())

before = len(df)
df = df.drop_duplicates(subset=content_cols, keep="first").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows. New shape: {df.shape}")

Duplicates after whitespace fix: 4
Removed 4 duplicate rows. New shape: (8786, 10)


### 3c. Convert hidden missing values to real nulls
Replacing `"Not Given"` with `NaN` lets Pandas treat these as missing. Missing directors are about 29% of the data, so **dropping those rows is not appropriate**: it would remove a large, non-random share of titles (most TV shows have no listed director). Instead:
- keep the rows
- add indicator columns so later tasks can tell what was missing
- fill with `"Unknown"` so text-based steps (grouping, recommendations) do not break

In [11]:
df = df.replace("Not Given", np.nan)

print("Missing after conversion:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing after conversion:
director    2587
country      287
dtype: int64


In [12]:
# How missing directors are spread across content type
pd.crosstab(df["type"], df["director"].isna(), normalize="index").round(3) * 100

director,False,True
type,,
Movie,97.2,2.8
TV Show,9.4,90.6


Missing directors are concentrated in TV Shows, which is expected (TV series often have many directors or none credited). This supports keeping the rows rather than dropping them.

In [13]:
df["director_missing"] = df["director"].isna()
df["country_missing"] = df["country"].isna()

df["director"] = df["director"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")

print("Remaining nulls:", int(df.isnull().sum().sum()))

Remaining nulls: 0


### 3d. Flag titles damaged by spreadsheet date conversion
Some titles were turned into date-like strings (pattern `day-Mon`). The original title cannot be recovered with certainty from this file alone, so the rows are kept and flagged rather than guessed.

In [14]:
date_like = df["title"].str.match(r"^\d{1,2}-[A-Z][a-z]{2}$")
df["title_format_issue"] = date_like
df.loc[date_like, ["show_id", "type", "title", "release_year", "country"]]

,show_id,type,title,release_year,country
393,s3997,TV Show,9-Feb,2018,Pakistan
2924,s3963,Movie,15-Aug,2019,India
3284,s4523,Movie,22-Jul,2018,Norway
4258,s5964,Movie,1-Oct,2014,Nigeria


### 3e. Validate the remaining formats
Check that dates parse, ratings are valid, and durations match content type.

In [15]:
# Every date_added should match month/day/year
parsed = pd.to_datetime(df["date_added"], format="%m/%d/%Y", errors="coerce")
print("Unparseable dates:", parsed.isna().sum())

# Ratings should be real certification codes, not durations like "74 min"
print("\nRating values:", sorted(df["rating"].unique()))

# Movies should be in minutes, TV shows in seasons
is_movie = df["type"] == "Movie"
print("\nMovies not in minutes:", (~df.loc[is_movie, "duration"].str.endswith("min")).sum())
print("TV shows not in seasons:", (~df.loc[~is_movie, "duration"].str.contains("Season")).sum())

Unparseable dates: 0

Rating values: ['G', 'NC-17', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'UR']

Movies not in minutes: 0
TV shows not in seasons: 0


All dates parse, all ratings are valid codes, and durations are consistent with type.

## Step 4: Transform categorical and date-related columns

### 4a. Date features

In [16]:
df["date_added"] = pd.to_datetime(df["date_added"], format="%m/%d/%Y")

df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month
df["month_name_added"] = df["date_added"].dt.month_name()
df["weekday_added"] = df["date_added"].dt.day_name()

# How old a title was when it reached Netflix
df["years_to_netflix"] = df["year_added"] - df["release_year"]

df[["title", "release_year", "date_added", "year_added", "years_to_netflix"]].head()

,title,release_year,date_added,year_added,years_to_netflix
0,Dick Johnson Is Dead,2020,2021-09-25,2021,1
1,Ganglands,2021,2021-09-24,2021,0
2,Midnight Mass,2021,2021-09-24,2021,0
3,Confessions of an Invisible Girl,2021,2021-09-22,2021,0
4,Sankofa,1993,2021-09-24,2021,28


In [17]:
# Sanity check: titles added before their release year
early = df[df["years_to_netflix"] < 0]
print("Titles added before release year:", len(early))
early["type"].value_counts()

Titles added before release year: 14


type
TV Show    12
Movie       2
Name: count, dtype: int64

**Interpretation:** most of these are TV Shows. For a series, `release_year` usually reflects its most recent season, while `date_added` is when the show first arrived on Netflix, so a negative gap is expected rather than an error. The few movies may be genuine data entry issues. All rows are kept and flagged so later tasks can decide how to treat them.

In [18]:
df["added_before_release"] = df["years_to_netflix"] < 0

### 4b. Split `duration` into numeric columns
`"90 min"` and `"2 Seasons"` cannot be averaged or plotted as text. Split them into a number and a unit, then into type-specific columns.

In [19]:
df["duration_value"] = df["duration"].str.extract(r"(\d+)").astype(int)
df["duration_unit"] = np.where(df["duration"].str.contains("min"), "min", "season")

df["movie_minutes"] = np.where(df["type"] == "Movie", df["duration_value"], np.nan)
df["tv_seasons"] = np.where(df["type"] == "TV Show", df["duration_value"], np.nan)

df[["type", "duration", "duration_value", "movie_minutes", "tv_seasons"]].sample(5, random_state=42)

,type,duration,duration_value,movie_minutes,tv_seasons
6441,Movie,130 min,130,130.0,NaN
7302,TV Show,1 Season,1,NaN,1.0
2372,Movie,78 min,78,78.0,NaN
3159,Movie,110 min,110,110.0,NaN
5778,TV Show,8 Seasons,8,NaN,8.0


### 4c. Group ratings into audience categories
There are 14 rating codes from two systems (TV ratings and MPAA film ratings). Grouping them by intended audience makes them comparable and easier to analyze.

In [20]:
rating_map = {
    "TV-Y": "Little Kids", "TV-G": "Little Kids", "G": "Little Kids",
    "TV-Y7": "Older Kids", "TV-Y7-FV": "Older Kids", "PG": "Older Kids", "TV-PG": "Older Kids",
    "PG-13": "Teens", "TV-14": "Teens",
    "R": "Adults", "TV-MA": "Adults", "NC-17": "Adults",
    "NR": "Unrated", "UR": "Unrated",
}
df["audience"] = df["rating"].map(rating_map)
print("Unmapped ratings:", df["audience"].isna().sum())
df["audience"].value_counts()

Unmapped ratings: 0


audience
Adults         4005
Teens          2645
Older Kids     1487
Little Kids     567
Unrated          82
Name: count, dtype: int64

### 4d. Genre features from `listed_in`
`listed_in` holds several genres in one string. Extract the primary (first-listed) genre and the number of genres per title. The full list is kept as a Python list for later tasks.

In [21]:
df["genres"] = df["listed_in"].str.split(", ")
df["primary_genre"] = df["genres"].str[0]
df["genre_count"] = df["genres"].str.len()

print("Unique genres across all titles:", df["genres"].explode().nunique())
df[["title", "listed_in", "primary_genre", "genre_count"]].head()

Unique genres across all titles: 42


,title,listed_in,primary_genre,genre_count
0,Dick Johnson Is Dead,Documentaries,Documentaries,1
1,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",Crime TV Shows,3
2,Midnight Mass,"TV Dramas, TV Horror, TV Mysteries",TV Dramas,3
3,Confessions of an Invisible Girl,"Children & Family Movies, Comedies",Children & Family Movies,2
4,Sankofa,"Dramas, Independent Movies, International Movies",Dramas,3


### 4e. Set categorical data types
Columns with a small, fixed set of values are stored as `category`, which uses less memory and makes intent clear.

In [22]:
audience_order = ["Little Kids", "Older Kids", "Teens", "Adults", "Unrated"]
df["audience"] = pd.Categorical(df["audience"], categories=audience_order, ordered=True)

for col in ["type", "rating", "duration_unit", "primary_genre", "month_name_added", "weekday_added"]:
    df[col] = df[col].astype("category")

df.dtypes

show_id                            str
type                          category
title                              str
director                           str
country                            str
date_added              datetime64[us]
release_year                     int64
rating                        category
duration                           str
listed_in                          str
director_missing                  bool
country_missing                   bool
title_format_issue                bool
year_added                       int32
month_added                      int32
month_name_added              category
weekday_added                 category
years_to_netflix                 int64
added_before_release              bool
duration_value                   int64
duration_unit                 category
movie_minutes                  float64
tv_seasons                     float64
audience                      category
genres                          object
primary_genre            

## Step 5: Create the clean dataset

In [ ]:
final_cols = [
    "show_id", "type", "title", "director", "country",
    "date_added", "year_added", "month_added", "month_name_added", "weekday_added",
    "release_year", "years_to_netflix",
    "rating", "audience",
    "duration", "duration_value", "duration_unit", "movie_minutes", "tv_seasons",
    "listed_in", "primary_genre", "genre_count",
    "director_missing", "country_missing", "title_format_issue", "added_before_release",
]
clean = df[final_cols].copy()

clean.to_csv("netflix_cleaned.csv", index=False)
print("Saved netflix_cleaned.csv with shape", clean.shape)

Saved netflix_cleaned.csv with shape (8786, 26)


Note: the `genres` list column is not saved because CSV stores lists as plain text. Recreate it in later tasks with `df["listed_in"].str.split(", ")`.

In [ ]:
# Before vs after summary
summary = pd.DataFrame({
    "Raw": [len(raw), raw.shape[1], int(raw.isnull().sum().sum()),
            int((raw == "Not Given").sum().sum()), int(raw.assign(title=raw["title"].str.replace("\u00a0", " ", regex=False).str.strip()).duplicated(subset=content_cols).sum())],
    "Clean": [len(clean), clean.shape[1], int(clean.isnull().sum().sum()) - int(clean[["movie_minutes", "tv_seasons"]].isnull().sum().sum()),
              0, int(clean.duplicated(subset=[c for c in clean.columns if c != "show_id"]).sum())],
}, index=["Rows", "Columns", "Null values (excluding by-design blanks)", "'Not Given' placeholders", "Duplicate records"])
summary

,Raw,Clean
Rows,8790,8786
Columns,10,26
Null values (excluding by-design blanks),0,0
'Not Given' placeholders,2875,0
Duplicate records,4,0


## Summary of cleaning decisions

| Issue found | Action taken | Reason |
|---|---|---|
| 4 duplicate titles under different IDs (1 hidden by whitespace) | Removed | Same content counted twice would inflate totals |
| Missing values hidden as "Not Given" (director ~29%, country ~3%) | Converted to null, flagged, filled with "Unknown" | Too many to drop; missingness is concentrated in TV Shows |
| Non-breaking and trailing spaces in 5 titles | Replaced and stripped (before deduplication) | Prevents identical values being treated as different |
| Titles converted to dates by a spreadsheet (e.g. "15-Aug") | Flagged, not changed | Original titles cannot be confirmed from this file |
| `date_added` stored as text | Converted to datetime; year, month and weekday extracted | Needed for trend analysis (Task 4) |
| Titles added before release year (mostly TV Shows) | Kept and flagged | Expected for multi-season shows |
| `duration` mixes minutes and seasons | Split into numeric value, unit, movie minutes and TV seasons | Allows numeric analysis by type |
| 14 rating codes across two systems | Grouped into 5 audience categories | Makes ratings comparable |
| Multiple genres in one field | Extracted primary genre and genre count | Supports EDA and classification (Tasks 2, 5) |

The cleaned file `netflix_cleaned.csv` is ready for Task 2 (EDA).